In [28]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2


The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import numpy as np
import math as math
import random as rand
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import matplotlib as mpl
import pylab
import pandas as pd
import statsmodels.api as sm
import bindata as bnd
import matplotlib.patches as mpatches
import matplotlib
from numpy import savetxt
from scipy.stats import multivariate_normal
from tqdm import tqdm

In [30]:
from kernel_NN import obs_Overlap, sqmmd_est2, row_Metric, row_mmDNN, mmDNN_cv, mmDNN_direct
from kernel_NN_snn import row_snn, snn_row_metric, snn_CV
from gendata import gendata_mcar, gendata_s_adopt

In [31]:
## Generate Data
expit = lambda x : np.exp(x)/(1 + np.exp(x))

def gendata_mcar(N, T, n, d, p, seed) : 
    """ 
    Generates Gaussian data, with latent dimension r = 2

    required : N, d are both EVEN positive integers
    """
    np.random.seed(seed = seed)

    ## Data Matrix (N * T * n * d)
    Data = np.zeros( (N, T, n, d) )
    true_Mean = np.zeros( (N, T, d) )
    true_Cov = np.zeros( (N, T, d, d) )

    u_1 = np.random.uniform(-1, 1, N)
    u_2 = np.random.uniform(0.2, 1, N)

    v_1 = np.random.uniform(-2, 2, T)
    v_2 = np.random.uniform(0.5, 2, T)

    even_ones = np.repeat([0, 1], d/2)
    odd_ones = np.repeat([1, 0], d/2)

    for i in range(N) : 
        for t in range(T) : 
            m_it = u_1[i]*v_1[t]*(even_ones - odd_ones)
            c_it = np.diag(u_2[i]*v_2[t]*(0.5*even_ones + odd_ones))
            true_Mean[i, t, :] = m_it
            true_Cov[i, t, :, :] = c_it
            dat_mat = np.random.multivariate_normal(m_it, c_it, size = n)
            Data[i, t, :, :] = dat_mat
    
    Masking = np.zeros( (N, T) )

    Masking = np.reshape(np.random.binomial(1, p, (N*T)), (N, T))
    
    return(Data, Masking, true_Mean, true_Cov)

def gendata_s_adopt(N, T, n, d, beta, seed) : 
    """ 
    Generates Gaussian data, with latent dimension r = 2

    beta : vector of two fractions between (0, 1)

    required : N, d are both EVEN positive integers
    """

    np.random.seed(seed = seed)

    Data = np.zeros( (N, T, n, d) )
    true_Mean = np.zeros( (N, T, d) )
    true_Cov = np.zeros( (N, T, d, d) )

    u_1 = np.random.uniform(-1, 1, N)
    u_2 = np.random.uniform(0.2, 1, N)

    v_1 = np.random.uniform(-2, 2, T)
    v_2 = np.random.uniform(0.5, 2, T)

    even_ones = np.repeat([0, 1], d/2)
    odd_ones = np.repeat([1, 0], d/2)

    for i in range(N) : 
        for t in range(T) : 
            m_it = u_1[i]*v_1[t]*(even_ones - odd_ones)
            c_it = np.diag(u_2[i]*v_2[t]*(0.5*even_ones + odd_ones))
            true_Mean[i, t, :] = m_it
            true_Cov[i, t, :, :] = c_it
            dat_mat = np.random.multivariate_normal(m_it, c_it, size = n)
            Data[i, t, :, :] = dat_mat

    Masking = np.zeros( (N, T) )
    pre_Masking = np.zeros( (N, T) )

    g1_inds = np.arange(0, N // 2)
    g2_inds = np.arange(N // 2, 3 * N // 4)
    g3_inds = np.arange(3 * N // 4, N)

    gamma_1 = [2, 0.7, 1, 0.7]
    gamma_2 = [2.5, 0.2, 1, 0.2]

    T1_lower = math.floor(T**beta[0])
    T2_lower = math.floor(T**beta[1])

    for i in range(N) : 
        if i in g1_inds :
            pre_Masking[i, :] = np.concatenate((np.ones(T1_lower), np.zeros(T - T1_lower)))
            for t in range(T - T1_lower) :
                pre_Masking[i, (t + T1_lower)] = np.random.binomial(1, expit(gamma_1[0] + ( 0.99**t )*gamma_1[1]*u_1[i-1] + gamma_1[2]*u_1[i] + ( 0.99**t )*gamma_1[3]*u_1[i+1]), 1)
            pre_A = pre_Masking[i, :]
            if len([i for i in range(len(pre_A)) if pre_A[i] == 0]) == 0:
                Masking[i, :] = pre_A
            elif len([i for i in range(len(pre_A)) if pre_A[i] == 0]) > 0:
                adopt_time = min([i for i in range(len(pre_A)) if pre_A[i] == 0]) 
                Masking[i, :] = np.concatenate((np.ones(adopt_time), np.zeros(T - adopt_time)))
        elif i in g2_inds :
            pre_Masking[i, :] = np.concatenate((np.ones(T2_lower), np.zeros(T - T2_lower)))
            for t in range(T - T2_lower) :
                pre_Masking[i, (t + T2_lower)] = np.random.binomial(1, expit(gamma_2[0] + ( 1.01**t )*gamma_2[1]*u_1[i-1] + gamma_2[2]*u_1[i] + ( 1.01**t )*gamma_2[3]*u_1[i+1]), 1)
            pre_A = pre_Masking[i, :]
            if len([i for i in range(len(pre_A)) if pre_A[i] == 0]) == 0:
                Masking[i, :] = pre_A
            elif len([i for i in range(len(pre_A)) if pre_A[i] == 0]) > 0:
                adopt_time = min([i for i in range(len(pre_A)) if pre_A[i] == 0]) 
                Masking[i, :] = np.concatenate((np.ones(adopt_time), np.zeros(T - adopt_time)))
            # adopt_time = min([i for i in range(len(pre_A)) if pre_A[i] == 0]) 
            # Masking[i, :] = np.concatenate((np.ones(adopt_time), np.zeros(T - adopt_time)))
        elif i in g3_inds : 
            Masking[i, :] = np.ones( T )

    return(Data, Masking, pre_Masking, true_Mean, true_Cov)


In [11]:
## Plot Settings
fix_plot_settings = True
if fix_plot_settings:
    plt.rc('font', family='serif')
    plt.rc('text', usetex=False)
    label_size = 20
    mpl.rcParams['xtick.labelsize'] = label_size 
    mpl.rcParams['ytick.labelsize'] = label_size 
    mpl.rcParams['axes.labelsize'] = label_size
    mpl.rcParams['axes.titlesize'] = label_size
    mpl.rcParams['figure.titlesize'] = label_size
    mpl.rcParams['lines.markersize'] = label_size
    mpl.rcParams['grid.linewidth'] = 2.5
    mpl.rcParams['legend.fontsize'] = label_size
    pylab.rcParams['xtick.major.pad']=5
    pylab.rcParams['ytick.major.pad']=5

    lss = ['--',  ':', '-.', '-', '--', '-.', ':', '-', '--', '-.', ':', '-']
    mss = ['>', 'o',  's', 'D', '>', 's', 'o', 'D', '>', 's', 'o', 'D']
    ms_size = [25, 20, 20, 20, 20, 20, 20, 20, 20, 20]
    colors = ['#e41a1c', '#0000cd', '#4daf4a',  'black' , 'magenta']
else:
    pass

In [ ]:
## Check Missingness by figure


N, T, n, d = 80, 80, 30, 2
kernel = "square"
beta = [3.9/6, 3.9/5]
eta_pool = np.arange(1, 25)/5
i, t = 0, T-1
sim = 1
p = 0.5

Data_stag, Masking_stag, pre_Masking_stag, true_Mean_stag, true_Cov_stag = gendata_s_adopt(N, T, n, d, beta, seed = sim)
Data, Masking, true_Mean, true_Cov = gendata_mcar(N, T, n, d, p, seed = sim)

# plt.matshow(Masking_stag)
# plt.matshow(Masking)

# plt.imshow(Masking, cmap = "Blues")

numb_ones = np.zeros(N)
for i in range(N):
    numb_ones[i] = sum(Masking_stag[i, :] == 1)
print(numb_ones)

numb_ones_sort = (np.sort(numb_ones))

Masking_stag_sort = np.zeros( (N, T) )
for i in range(N) : 
    Masking_stag_sort[i, :] = np.concatenate( (np.ones(int(numb_ones_sort[i])), np.zeros(int(T - numb_ones_sort[i]))) )



# plt.imshow(Masking_stag_sort, cmap = "Blues")
# plt.axis('off')
# plt.show()

plt.imshow(Masking, cmap = "Blues")
# plt.colorbar()
# plt.legend(['Observed', 'Missing'])
# plt.xticks()
# plt.yticks()
plt.axis('off')
plt.show()

In [ ]:
## Simulation: Staggered Adoption 
## Will proceed by taking d = 2, 4, 8...

T, n, d = 80, 30, 4
beta = [3.9/6, 3.9/5]
kernel = "square"
# eta_pool = np.arange(1, 30, 0.5)/3
eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) )
i, t = 0, T - 1

nsim = 30

pools = []
pools_eta = []
# print(f"{0}-th iteration")
for i, row_exp in enumerate(np.arange(5, 9)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    perf_pool, eta_star_pool = np.zeros( nsim ), np.zeros( nsim )

    for sim in tqdm(range(nsim)) :
        Data, Masking, pre_Masking, true_Mean, true_Cov = gendata_s_adopt(N, T, n, d, beta, seed = sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]

        row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)

        eta_star = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        eta_star_pool[sim] = eta_star

        hat_mu_it = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star)
        samples_from_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = (N*n) )
        perf_pool[sim] = sqmmd_est2( hat_mu_it, samples_from_truth, kernel )
    pools_eta.append(eta_star_pool)
    pools.append(perf_pool)    

with open("results/stagreal_80_30_4.pkl", "wb") as f:
    pickle.dump(pools, f)

with open("results/stagreal_80_30_4_eta.pkl", "wb") as f:
    pickle.dump(pools_eta, f)

In [ ]:
## Choosing tuning parameter candidates -- after running a simple trial, localize the grid

eta_pool = np.arange(1, 25, 0.5)/6
print(eta_pool)
print(len(eta_pool))

# eta_pool2 = np.arange(22, 50, 1.5)/5
# print(eta_pool2)

eta_pool_final = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) )
print(eta_pool_final)
print(len(eta_pool_final))

In [ ]:
## Simulation: MCAR 

T, n, d = 80, 30, 6
kernel = "square"
# eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) )
eta_pool = np.concatenate( ( np.arange(1, 25, 1)/6, np.arange(22, 50, 0.6)/5 ) )
i, t = 0, T - 1
p = 0.5

nsim = 30

pools = []
pools_eta = []
# print(f"{0}-th iteration")
for i, row_exp in enumerate(np.arange(4, 8)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    perf_pool, eta_star_pool = np.zeros( nsim ), np.zeros( nsim )

    for sim in tqdm(range(nsim)) :
        Data, Masking, true_Mean, true_Cov = gendata_mcar(N, T, n, d, p, seed = sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]

        row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)

        eta_star = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        eta_star_pool[sim] = eta_star

        hat_mu_it = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star)
        samples_from_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = (N*n) )
        perf_pool[sim] = sqmmd_est2( hat_mu_it, samples_from_truth, kernel )
    pools_eta.append(eta_star_pool)
    pools.append(perf_pool)    

with open("results/mcarreal_80_30_6_p05.pkl", "wb") as f:
    pickle.dump(pools, f)

with open("results/mcarreal_80_30_6_p05_eta.pkl", "wb") as f:
    pickle.dump(pools_eta, f)

In [ ]:
## Simulation: SNN vs. DNN under MCAR (Comparing the mean)

expit = lambda x : np.exp(x)/(1 + np.exp(x))

def gendata_mcar(N, T, n, d, p, seed) : 
    """ 
    Generates Gaussian data, with latent dimension r = 2

    required : N, d are both EVEN positive integers
    """
    np.random.seed(seed = seed)

    ## Data Matrix (N * T * n * d)
    Data = np.zeros( (N, T, n, d) )
    true_Mean = np.zeros( (N, T, d) )
    true_Cov = np.zeros( (N, T, d, d) )

    u_1 = np.random.uniform(-1, 1, N)
    u_2 = np.random.uniform(0.2, 1, N)

    v_1 = np.random.uniform(-2, 2, T)
    v_2 = np.random.uniform(0.5, 2, T)

    even_ones = np.repeat([0, 1], d/2)
    odd_ones = np.repeat([1, 0], d/2)

    for i in range(N) : 
        for t in range(T) : 
            m_it = u_1[i]*v_1[t]*(even_ones - odd_ones)
            c_it = np.diag(u_2[i]*v_2[t]*(0.5*even_ones + odd_ones))
            true_Mean[i, t, :] = m_it
            true_Cov[i, t, :, :] = c_it
            dat_mat = np.random.multivariate_normal(m_it, c_it, size = n)
            Data[i, t, :, :] = dat_mat
    
    Masking = np.zeros( (N, T) )

    Masking = np.reshape(np.random.binomial(1, p, (N*T)), (N, T))
    
    return(Data, Masking, true_Mean, true_Cov)

T, n, d = 80, 30, 6
kernel = "square"
eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) ) 
eta_pool_snn = np.concatenate((np.arange(0, 1, 0.05), np.arange(1, 3, 0.4)))
i, t = 0, T - 1
p = 0.5

nsim = 30

pools = []
pools_eta = []
pools_eta_snn = []
pools_snn_ests = []
pools_mmd_mean_ests = []
pools_realmean_ests = []

for i, row_exp in enumerate(np.arange(5, 9)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    perf_pool, eta_star_pool = np.zeros( nsim ), np.zeros( nsim )
    perf_snn_pool, eta_star_snn_pool = np.zeros( nsim ), np.zeros( nsim )
    snn_ests = np.zeros([nsim, d])
    # snn_ests_sd = np.zeros([nsim, d])
    # snn_ests_quantile = np.zeros([nsim, d])
    mmd_mean_ests = np.zeros([nsim, d])
    real_mean = np.zeros([nsim, d])

    for sim in tqdm(range(nsim)) :
        Data, Masking, true_Mean, true_Cov = gendata_mcar(N, T, n, d, p, sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]
        Data_mean = np.mean(Data, axis = 2)
        Data_std = np.std(Data, axis = 2)
        Data_quantile = np.quantile(Data, q = 0.1,axis = 2)
        row_Dissim_vec = np.zeros(N)
        snn_row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)
            snn_row_Dissim_vec[j] = snn_row_metric(i, j, t, Data_mean, Masking, exc_opt = True)
        eta_star_snn = snn_CV(Data_mean, Masking, eta_cand = eta_pool_snn)
        eta_star = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        eta_star_pool[sim] = eta_star
        eta_star_snn_pool[sim] = eta_star_snn

        hat_mu_it = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star)
        neighbors_it = row_snn(i, t, Data_mean, snn_row_Dissim_vec, Masking, eta = eta_star_snn)
        mean_it = np.mean(neighbors_it, axis = 0)
        mmd_mean_ests[sim] = np.mean(hat_mu_it, axis = 0)
        snn_ests[sim] = mean_it
        real_mean[sim] = true_Mean_it
        samples_from_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = (N*n) )
        perf_pool[sim] = sqmmd_est2( hat_mu_it, samples_from_truth, kernel )
    pools_eta.append(eta_star_pool)
    pools.append(perf_pool)
    pools_snn_ests.append(snn_ests)
    pools_mmd_mean_ests.append(mmd_mean_ests)
    pools_realmean_ests.append(real_mean)
    pools_eta_snn.append(eta_star_snn_pool)

np.save("mcar80_30_2.npy", np.array(pools, dtype = object), allow_pickle=True)
np.save("mcar80_30_2_eta.npy", np.array(pools_eta, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_snn_ests.npy", np.array(pools_snn_ests, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_mmd_mean_ests.npy", np.array(pools_mmd_mean_ests, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_realmean_ests.npy", np.array(pools_realmean_ests, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_eta_snn.npy", np.array(pools_eta_snn, dtype = object), allow_pickle = True)


In [ ]:
## Simulation: SNN vs. DNN under MCAR (Comparing marginal quantiles)

expit = lambda x : np.exp(x)/(1 + np.exp(x))

def gendata_mcar(N, T, n, d, p, seed) : 
    """ 
    Generates Gaussian data, with latent dimension r = 2

    required : N, d are both EVEN positive integers
    """
    np.random.seed(seed = seed)

    ## Data Matrix (N * T * n * d)
    Data = np.zeros( (N, T, n, d) )
    true_Mean = np.zeros( (N, T, d) )
    true_Cov = np.zeros( (N, T, d, d) )

    u_1 = np.random.uniform(-1, 1, N)
    u_2 = np.random.uniform(0.2, 1, N)

    v_1 = np.random.uniform(-2, 2, T)
    v_2 = np.random.uniform(0.5, 2, T)

    even_ones = np.repeat([0, 1], d/2)
    odd_ones = np.repeat([1, 0], d/2)

    for i in range(N) : 
        for t in range(T) : 
            m_it = u_1[i]*v_1[t]*(even_ones - odd_ones)
            c_it = np.diag(u_2[i]*v_2[t]*(0.5*even_ones + odd_ones))
            true_Mean[i, t, :] = m_it
            true_Cov[i, t, :, :] = c_it
            dat_mat = np.random.multivariate_normal(m_it, c_it, size = n)
            Data[i, t, :, :] = dat_mat
    
    Masking = np.zeros( (N, T) )

    Masking = np.reshape(np.random.binomial(1, p, (N*T)), (N, T))
    
    return(Data, Masking, true_Mean, true_Cov)

T, n, d = 80, 30, 2
kernel = "square"
eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) ) 
eta_pool_snn = np.concatenate((np.arange(0, 1, 0.05), np.arange(1, 3, 0.4)))
i, t = 0, T - 1
p = 0.5

nsim = 30

pools = []
pools_eta = []
pools_eta_snn = []
pools_snn_ests = []
pools_mmd_quantile_ests = []
pools_realquantile_ests = []

for i, row_exp in enumerate(np.arange(5, 9)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    perf_pool, eta_star_pool = np.zeros( nsim ), np.zeros( nsim )
    perf_snn_pool, eta_star_snn_pool = np.zeros( nsim ), np.zeros( nsim )
    snn_ests = np.zeros([nsim, d])
    mmd_quantile_ests = np.zeros([nsim, d])
    real_quantile = np.zeros([nsim, d])

    for sim in tqdm(range(nsim)) :
        Data, Masking, true_Mean, true_Cov = gendata_mcar(N, T, n, d, p, sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]
        Data_quantile = np.quantile(Data, q = 0.1, axis = 2)
        row_Dissim_vec = np.zeros(N)
        snn_row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)
            snn_row_Dissim_vec[j] = snn_row_metric(i, j, t, Data_quantile, Masking, exc_opt = True)
        eta_star_snn = snn_CV(Data_quantile, Masking, eta_cand = eta_pool_snn)
        eta_star = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        eta_star_pool[sim] = eta_star
        eta_star_snn_pool[sim] = eta_star_snn

        hat_mu_it = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star)
        neighbors_it = row_snn(i, t, Data_quantile, snn_row_Dissim_vec, Masking, eta = eta_star_snn)
        mmd_quantile_ests[sim] = np.quantile(hat_mu_it, q = 0.1, axis = 0)
        snn_ests[sim] = np.mean(neighbors_it, axis = 0)
        samples_from_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = 10**4 )
        perf_pool[sim] = sqmmd_est2( hat_mu_it, samples_from_truth, kernel )
        real_quantile[sim] = np.quantile( samples_from_truth, q = 0.1, axis = 0 )
    pools_eta.append(eta_star_pool)
    pools_eta_snn.append(eta_star_snn_pool)
    pools.append(perf_pool)
    pools_snn_ests.append(snn_ests)
    pools_mmd_quantile_ests.append(mmd_quantile_ests)
    pools_realquantile_ests.append(real_quantile)

np.save("mcar80_30_2_q01_new.npy", np.array(pools, dtype = object), allow_pickle=True)
np.save("mcar80_30_2_q01_eta_new.npy", np.array(pools_eta, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_q01_eta_snn_new.npy", np.array(pools_eta_snn, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_q01_snn_ests_new.npy", np.array(pools_snn_ests, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_q01_mmd_mean_ests_new.npy", np.array(pools_mmd_quantile_ests, dtype = object), allow_pickle = True)
np.save("mcar80_30_2_q01_realquantile_ests_new.npy", np.array(pools_realquantile_ests, dtype = object), allow_pickle = True)


In [32]:
### Simulation: CV vs. direct under staggered adoption 

T, n, d = 80, 30, 4
beta = [3.9/6, 3.9/5]
kernel = "square"
eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) )
delta = 1/2
i, t = 0, T - 1
nsim = 2

## Final output of simulation: 
neighbor_kernelNN_cv = [] 
neighbor_kernelNN_direct = []
observed = []
target = []
eta_star_cv = []
eta_star_direct = []

# ## Performance measured by MMD_k, where k is used for algorithm
# mmd_perfs = [] # (1) by 'row-size N' (2) by 'simulation rounds'
# ## Performance measured by KS statistics
# ks_perfs = [] # (1) by 'row-size N' (2) by 'simulation rounds'
# ## Optimal eta 
# eta_stars = [] # (1) by 'row-size N' (2) by 'simulation rounds'

for i, row_exp in enumerate(np.arange(5, 9)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    neighbor_cv_N = []
    neighbor_direct_N = []
    observed_N = []
    target_N = []
    eta_star_cv_N = []
    eta_star_direct_N = []
    
    for sim in tqdm(range(nsim)) :
        Data, Masking, pre_Masking, true_Mean, true_Cov = gendata_s_adopt(N, T, n, d, beta, seed = sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]

        row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)

        ## Choose optimal eta 
        eta_star_cv_Niter = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        # eta_direct_pool, perf_direct_pool = [], []
        # for delta in delta_pool:
        #     eta_direct, perf_direct = mmDNN_direct(i, t, Data, row_Dissim_vec, Masking, eta_pool, delta, kernel)
        #     eta_direct_pool.append(eta_direct)
        #     perf_direct_pool.append(perf_direct)
        # eta_star_direct = eta_direct_pool[np.argmin(perf_direct_pool)]
        eta_star_direct_Niter = mmDNN_direct(i, t, Data, row_Dissim_vec, Masking, eta_pool, delta, kernel)

        eta_star_cv_N.append(eta_star_cv_Niter)
        eta_star_direct_N.append(eta_star_direct_Niter)

        ## Find neighbors given optimal eta
        neighbor_cv_Niter = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star_cv_Niter)
        neighbor_direct_Niter = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star_direct_Niter)

        neighbor_cv_N.append(neighbor_cv_Niter)
        neighbor_direct_N.append(neighbor_direct_Niter)

        ## Observed data from the target
        target_Niter = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = 10**5 ) # The 'density' of the truth
        observed_Niter = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = n ) # Samples from the truth

        target_N.append(target_Niter)
        observed_N.append(observed_Niter)

        # perf_pool_cv[sim] = sqmmd_est2( neighbor_cv, pseudo_truth, kernel )
        # perf_pool_direct[sim] = sqmmd_est2( neighbor_direct, pseudo_truth, kernel )
    
    neighbor_kernelNN_cv.append(neighbor_cv_N)
    neighbor_kernelNN_direct.append(neighbor_direct_N)
    observed.append(observed_N)
    target.append(target_N)
    eta_star_cv.append(eta_star_cv_N)
    eta_star_direct.append(eta_star_direct_N)


with open("final_round/stag_80_30_4_neighbor_cv.pkl", "wb") as f:
    pickle.dump(neighbor_kernelNN_cv, f)
with open("results/stagreal_80_30_4_neighbor_direct.pkl", "wb") as f:
    pickle.dump(neighbor_kernelNN_direct, f)
with open("results/stagreal_80_30_4_observed.pkl", "wb") as f:
    pickle.dump(observed, f)
with open("results/stagreal_80_30_4_target.pkl", "wb") as f:
    pickle.dump(target, f)
with open("results/stagreal_80_30_4_eta_cv.pkl", "wb") as f:
    pickle.dump(eta_star_cv, f)
with open("results/stagreal_80_30_4_eta_direct.pkl", "wb") as f:
    pickle.dump(eta_star_direct, f)

0-th iteration


  0%|          | 0/2 [00:00<?, ?it/s]/var/folders/ng/1zd7d3bs0p79qr4lw85bcbxh0000gn/T/ipykernel_4580/1019269970.py:91: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pre_Masking[i, (t + T1_lower)] = np.random.binomial(1, expit(gamma_1[0] + ( 0.99**t )*gamma_1[1]*u_1[i-1] + gamma_1[2]*u_1[i] + ( 0.99**t )*gamma_1[3]*u_1[i+1]), 1)
/var/folders/ng/1zd7d3bs0p79qr4lw85bcbxh0000gn/T/ipykernel_4580/1019269970.py:101: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pre_Masking[i, (t + T2_lower)] = np.random.binomial(1, expit(gamma_2[0] + ( 1.01**t )*gamma_2[1]*u_1[i-1] + gamma_2[2]*u_1[i] + ( 1.01**t )*gamma_2[3]*u_1[i+1]), 1)
100%|██████████| 2/2 [

1-th iteration


  0%|          | 0/2 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [6]:
## Simulation: CV vs. direct under staggered adoption 

T, n, d = 80, 30, 4
beta = [3.9/6, 3.9/5]
kernel = "square"
eta_pool = np.concatenate( ( np.arange(1, 25, 0.5)/6, np.arange(22, 50, 1.5)/5 ) )
delta_pool = [1/4, 1/2]
i, t = 0, T - 1
nsim = 2

pools = []
pools_eta = []

for i, row_exp in enumerate(np.arange(5, 9)): 
    print(f"{i}-th iteration")
    N = 2**(row_exp)
    perf_pool_cv, eta_star_cv_pool, perf_pool_direct, eta_star_direct_pool = np.zeros( nsim ), np.zeros( nsim ), np.zeros( nsim ), np.zeros( nsim )

    for sim in tqdm(range(nsim)) :
        Data, Masking, pre_Masking, true_Mean, true_Cov = gendata_s_adopt(N, T, n, d, beta, seed = sim)
        true_Mean_it = true_Mean[i, t, :]
        true_Cov_it = true_Cov[i, t, :, :]

        row_Dissim_vec = np.zeros(N)

        for j in range(N) :
            row_Dissim_vec[j] = row_Metric(i, j, t, Data, Masking, kernel, exc_opt = True)

        eta_star_cv = mmDNN_cv(Data, Masking, kernel, eta_cand = eta_pool)
        eta_direct_pool, perf_direct_pool = [], []
        for delta in delta_pool:
            eta_direct, perf_direct = mmDNN_direct(i, t, Data, row_Dissim_vec, Masking, eta_pool, delta, kernel)
            eta_direct_pool.append(eta_direct)
            perf_direct_pool.append(perf_direct)
        eta_star_direct = eta_direct_pool[np.argmin(perf_direct_pool)]
    
        hat_mu_it_cv = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star_cv)
        hat_mu_it_direct = row_mmDNN(i, t, Data, row_Dissim_vec, Masking, eta = eta_star_direct)
        pseudo_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = 10**5 ) # The 'density' of the truth
        samples_from_truth = np.random.multivariate_normal( true_Mean_it, true_Cov_it, size = n ) # Samples from the truth

    

0-th iteration


  0%|          | 0/2 [00:00<?, ?it/s]/var/folders/ng/1zd7d3bs0p79qr4lw85bcbxh0000gn/T/ipykernel_3761/1019269970.py:91: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pre_Masking[i, (t + T1_lower)] = np.random.binomial(1, expit(gamma_1[0] + ( 0.99**t )*gamma_1[1]*u_1[i-1] + gamma_1[2]*u_1[i] + ( 0.99**t )*gamma_1[3]*u_1[i+1]), 1)
/var/folders/ng/1zd7d3bs0p79qr4lw85bcbxh0000gn/T/ipykernel_3761/1019269970.py:101: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pre_Masking[i, (t + T2_lower)] = np.random.binomial(1, expit(gamma_2[0] + ( 1.01**t )*gamma_2[1]*u_1[i-1] + gamma_2[2]*u_1[i] + ( 1.01**t )*gamma_2[3]*u_1[i+1]), 1)
100%|██████████| 2/2 [

1-th iteration


100%|██████████| 2/2 [00:35<00:00, 17.83s/it]


2-th iteration


100%|██████████| 2/2 [03:01<00:00, 90.53s/it] 


3-th iteration


100%|██████████| 2/2 [20:48<00:00, 624.48s/it]


In [10]:
grand_lst = []

lst1 = []
lst2 = []

lst1.append([1, 2])
lst1.append([1])

lst2.append([4, 5, 6, 7])
lst2.append([4, 5, 6, 0, 0])
lst2.append([2])

# emp_lst.append([1, 2])
# emp_lst.append([4, 5, 6, 7])
# print(emp_lst)

grand_lst.append(lst1)
grand_lst.append(lst2)

print(grand_lst)

print(grand_lst[1])

[[[1, 2], [1]], [[4, 5, 6, 7], [4, 5, 6, 0, 0], [2]]]
[[4, 5, 6, 7], [4, 5, 6, 0, 0], [2]]
